# Interactive Agent Session: Chapter 5 — Dify Headless Knowledge Concierge

In this session, we transition from "Coding Agents" to **"Platform Orchestration."** 

**Dify** (Designable LLM Application Platform) allows No-Code architects to build complex agents with pre-configured Knowledge Bases (RAG), Triggers, and Logic. Our job as developers is to treat this entire platform as a **Headless API.**

**The Business Goal:** Integrate a Dify-powered "Sales Intelligence" bot into a custom corporate dashboard. 

**Key Features:**
1. **REST API Integration**: Calling Dify as a standardized backend service.
2. **Stateful Conversation Management**: Tracking the `conversation_id` across client-side turns.
3. **Decoupled Architecture**: Showing how a PM can update agent logic in Dify *without* the developer changing a single line of Python code.

**Prerequisites:** Set your `DIFY_API_KEY` in your `.env` file.

In [ ]:
import os
import requests
from dotenv import load_dotenv
from IPython.display import display, HTML

load_dotenv()

# ── 0. Configuration ──────────────────────────────────────────────────────────
DIFY_API_KEY = os.getenv("DIFY_API_KEY")
DIFY_ENDPOINT = "https://api.dify.ai/v1/chat-messages"

if not DIFY_API_KEY or DIFY_API_KEY == "your_dify_api_key_here":
    USE_SIMULATION = True
    print("⚠️  No Dify API Key found — running in high-fidelity simulation mode.")
else:
    USE_SIMULATION = False
    print("✅ Dify API Client Initialized.")

# ── 1. The Headless Client Wrapper ───────────────────────────────────────────
def query_dify_agent(user_query: str, conversation_id: str = ""):
    if USE_SIMULATION:
        # Simulate a high-quality Dify RAG response
        return {
            "answer": "Based on our internal pricing guide (Retrieved via Dify Knowledge), for a team of 45 people, you qualify for the **Enterprise Tier**. This includes 24/7 dedicated support and unlimited workspace seats. Should I schedule a demo?",
            "conversation_id": "conv_sim_999"
        }
    
    headers = {
        "Authorization": f"Bearer {DIFY_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "inputs": {},
        "query": user_query,
        "response_mode": "blocking",
        "conversation_id": conversation_id,
        "user": "admin_user_001"
    }
    
    try:
        response = requests.post(DIFY_ENDPOINT, headers=headers, json=payload)
        response.raise_for_status()
        data = response.json()
        return {
            "answer": data.get('answer', 'No answer received.'),
            "conversation_id": data.get('conversation_id', '')
        }
    except Exception as e:
        return {"answer": f"Dify Bridge Error: {e}", "conversation_id": ""}

# ── 2. Premium Knowledge Terminal UI ────────────────────────────────────────
def render_dify_ui(query: str, response: dict):
    html = f'''
    <style>
        @import url("https://fonts.googleapis.com/css2?family=Unbounded:wght@400;700&family=Space+Grotesk:wght@300;400;700&display=swap");
        .dify-terminal {{ max-width: 850px; margin: 30px auto; font-family: "Space Grotesk", sans-serif; background: #080a0f; padding: 45px; border-radius: 40px; border: 1px solid #1a1e2e; box-shadow: 0 60px 120px rgba(0,0,0,0.8); color: #fff; }}
        .dify-header {{ display: flex; align-items: center; justify-content: space-between; margin-bottom: 40px; }}
        .dify-logo {{ font-family: "Unbounded", sans-serif; font-size: 20px; font-weight: 700; color: #6366f1; }}
        .dify-badge {{ background: #ffffff0a; color: #94a3b8; padding: 6px 14px; border-radius: 50px; font-size: 10px; text-transform: uppercase; font-weight: 800; border: 1px solid #ffffff14; }}
        .dify-query {{ background: #131722; padding: 25px; border-radius: 20px; margin-bottom: 25px; border-left: 4px solid #6366f1; }}
        .dify-label {{ font-size: 10px; color: #6366f1; text-transform: uppercase; font-weight: 800; margin-bottom: 10px; display: block; }}
        .dify-answer {{ background: #ffffff; color: #0f172a; padding: 35px; border-radius: 24px; font-size: 16px; line-height: 1.8; position: relative; overflow: hidden; }}
        .dify-answer:before {{ content: ""; position: absolute; top:0; left:0; width: 100%; height: 5px; background: linear-gradient(90deg, #6366f1, #a855f7); }}
        .dify-metadata {{ margin-top: 20px; font-size: 11px; color: #475569; font-family: monospace; }}
    </style>
    <div class="dify-terminal">
        <div class="dify-header">
            <div class="dify-logo">Dify // API BRIDGE</div>
            <div class="dify-badge">Cloud-Orchestrated Agent Active</div>
        </div>
        
        <div class="dify-query">
            <span class="dify-label">Incoming Request</span>
            <div style="font-size: 18px; font-weight: 400;">{query}</div>
        </div>

        <div class="dify-answer">
            <span class="dify-label" style="color: #6366f1; opacity: 0.6">Headless Agent Response</span>
            {response['answer']}
        </div>

        <div class="dify-metadata">
            CONVERSATION_ID: {response['conversation_id']} | STATUS: SYNCED
        </div>
    </div>
    '''
    display(HTML(html))

# ── 3. Execution ─────────────────────────────────────────────────────────────
USER_INPUT = "What is the pricing for a team of 45 people?"
result = query_dify_agent(USER_INPUT)
render_dify_ui(USER_INPUT, result)
